# Splane Mouse Hypothalamus (1/2): Identify uniform spatial domain

## Data preprocessing

In [3]:
from SPACEL.setting import set_environ_seed
seed = 13
set_environ_seed(seed)
from SPACEL import Splane
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib
import anndata
import glob
import os
from natsort import natsorted

Setting environment seed: 13


Here, we will incorporate the cell type composition predicted by **Spoint** into the spatial anndata object for subsequent spatial domain identification in **Splane** using the `add_cell_type_composition` function. This function takes a DataFrame containing the cell type composition matrix as input for spot-based spatial transcriptomic data or a series of cell type annotations as input for single-cell resolution spatial transcriptomic data.

In [ ]:
# Path to the directory
data_dir = './embryo'

# Find all .h5ad files
h5ad_files = natsorted(glob.glob(os.path.join(data_dir, '*.h5ad')))
# h5ad_files
adata_list = []
# Read all .h5ad files into a list of AnnData objects
for f in h5ad_files:
    ad = sc.read(f)
    adata_list.append(ad)

Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].


In [5]:
ad.obs['annotation'].unique()

['Head mesenchyme', 'Inner ear', 'Cavity', 'Connective tissue', 'Mesenchyme', ..., 'Jaw and tooth', 'Heart', 'Surface ectoderm', 'Liver', 'Cartilage primordium']
Length: 19
Categories (19, object): ['Brain', 'Branchial arch', 'Cartilage primordium', 'Cavity', ..., 'Sclerotome', 'Surface ectoderm', 'Sympathetic nerve', 'Urogenital ridge']

In [6]:
# Collect all unique cell types across all .h5ad files
all_cell_types = set()
for adata in adata_list:
    all_cell_types.update(adata.obs['annotation'].unique().tolist())

all_cell_types = sorted(list(all_cell_types)) 
# all_cell_types

In [7]:
for i, adata in enumerate(adata_list):    
    # Add cell type composition
    Splane.utils.add_cell_type_composition(adata, celltype_anno=adata.obs['annotation'])
    # Add missing cell type columns as 0s
    for ct in all_cell_types:
        if ct not in adata.obs.columns:
            adata.obs[ct] = 0.0
            
    # Reorder columns consistently
    ordered_cols = [c for c in adata.obs.columns if c not in all_cell_types] + all_cell_types
    adata.obs = adata.obs[ordered_cols]
    adata.obs['slice_id'] = i
    adata.uns['celltypes'] = np.array(all_cell_types, dtype=object)

In [8]:
adata_list[1].obs

,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,annotation,Regulon - Ahctf1,Regulon - Alx1,Regulon - Alx3,Regulon - Alx4,Regulon - Arx,...,Lung primordium,Meninges,Mesenchyme,Pancreas primordium,Sclerotome,Spinal cord,Surface ectoderm,Sympathetic nerve,Urogenital ridge,slice_id
cell_name,,,,,,,,,,,,,,,,,,,,,
132_345,452,6.115892,809,6.697034,Cavity,0.068070,0.040164,0.166280,0.097237,0.002689,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
132_346,945,6.852243,2005,7.603898,Mesenchyme,0.128876,0.022347,0.066029,0.081802,0.028202,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
132_347,828,6.720220,1626,7.394493,Mesenchyme,0.009992,0.063498,0.091322,0.164242,0.005345,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
132_348,978,6.886532,2059,7.630461,Mesenchyme,0.023304,0.059755,0.078087,0.167988,0.005496,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
132_349,965,6.873164,2001,7.601902,Mesenchyme,0.223084,0.065970,0.084648,0.202689,0.008216,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315_372,1627,7.395108,4396,8.388678,Surface ectoderm,0.148242,0.016430,0.030432,0.022183,0.006707,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1
315_373,1228,7.113956,3085,8.034631,Surface ectoderm,0.106832,0.025916,0.060321,0.052512,0.005833,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1
315_374,892,6.794587,1992,7.597396,Surface ectoderm,0.000000,0.003189,0.054794,0.030323,0.009208,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1


## Training Splane model

In this step, we initialize the Splane model by ``Splane.init_model(...)`` using the anndata object list as input. The ``n_clusters`` parameter determines the number of spatial domains to be identified. The ``k`` parameter controls the degree of neighbors considered in the model, with a larger ``k`` value resulting in more emphasis on global structure rather than local structure. The ``gnn_dropout`` parameter influences the level of smoothness in the model’s predictions, with a higher ``gnn_dropout`` value resulting in a smoother output that accommodates the sparsity of the spatial transcriptomics data. 

We train the model by ``splane.train(...)`` to obtain latent feature of each spots/cells. The parameter ``d_l`` affects the level of batch effect correction between slices. By default, ``d_l`` is ``0.2`` for spatial transcriptomics data with single cell resolution.

Then, we can identify the spatial domain to which each spot/cell belongs by ``splane.identify_spatial_domain(...)``. By default, the results will be saved in ``spatial_domain`` column in ``.obs``. If the key parameter is provided, the results will be saved in ``.obs[key]``.

In [ ]:
splane_model = Splane.init_model(adata_list, n_clusters=25,use_gpu=False,n_neighbors=25, gnn_dropout=0.5, seed=seed)
splane_model.train(d_l=0.2)
splane_model.identify_spatial_domain()
    
sc.concat(adata_list).write(f'./{seed}/embryo_11.5_S1_S4_Splane.h5ad')    

Setting global seed: 13
Calculating cell type weights...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/utils.py:76: RuntimeWarning: invalid value encountered in double_scalars
  return len(x) / W * inum / z2ss


Generating GNN inputs...
Calculating largest eigenvalue of normalized graph Laplacian...
Calculating Chebyshev polynomials up to order 2...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/pygcn_utils.py:57: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:641.)
  return torch.sparse.FloatTensor(indices, values, shape)


Setting the weight of similarity loss to 1.776


The best epoch 140 total loss=-2.729 g loss=-2.471 d loss=1.290 d acc=0.455 simi loss=-0.984 db loss=0.735:  65%|██████▌   | 195/300 [37:29<20:11, 11.54s/it]

Stop trainning because of loss convergence



/hpc/home/fl135/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [10]:
adata_list[0]

AnnData object with n_obs × n_vars = 30124 × 26854
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'annotation', 'Regulon - A1cf', 'Regulon - Acaa1a', 'Regulon - Adnp', 'Regulon - Alx1', 'Regulon - Alx3', 'Regulon - Alx4', 'Regulon - Arid3a', 'Regulon - Arid3c', 'Regulon - Arid5b', 'Regulon - Arx', 'Regulon - Ascl2', 'Regulon - Atf2', 'Regulon - Atf5', 'Regulon - Atf6', 'Regulon - Atf6b', 'Regulon - Atoh1', 'Regulon - Barhl1', 'Regulon - Barhl2', 'Regulon - Barx2', 'Regulon - Bcl3', 'Regulon - Bclaf1', 'Regulon - Bhlhe22', 'Regulon - Bmyc', 'Regulon - Brf1', 'Regulon - Brf2', 'Regulon - Bsx', 'Regulon - Cdx2', 'Regulon - Cdx4', 'Regulon - Cebpa', 'Regulon - Cebpb', 'Regulon - Cebpd', 'Regulon - Cebpe', 'Regulon - Cebpg', 'Regulon - Chd1', 'Regulon - Cic', 'Regulon - Clock', 'Regulon - Cpeb1', 'Regulon - Creb1', 'Regulon - Creb3', 'Regulon - Creb3l1', 'Regulon - Creb3l2', 'Regulon - Crem', 'Regulon - Ctcf', 'Regulon - Cux1', 'Regulon - Cux2